In [2]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

print("Python, TF versions:")
import sys
print(sys.version)
print("TensorFlow:", tf.__version__)

# GPU setup (optional)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs available: {len(gpus)}")
    except Exception as e:
        print("GPU config error:", e)
else:
    print("No GPU detected.")

# Mixed precision if GPU available and beneficial
try:
    from tensorflow.keras import mixed_precision
    if gpus:
        mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision enabled (policy: mixed_float16).")
    else:
        print("Mixed precision skipped (no GPU).")
except Exception as e:
    print("Mixed precision not configured:", e)

# Reproducibility seeds (note: fully deterministic training is hard)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


Python, TF versions:
3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
TensorFlow: 2.20.0
No GPU detected.
Mixed precision skipped (no GPU).


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

print("Python, TF versions:")
import sys
print(sys.version)
print("TensorFlow:", tf.__version__)

# GPU setup (optional)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs available: {len(gpus)}")
    except Exception as e:
        print("GPU config error:", e)
else:
    print("No GPU detected.")

# Mixed precision if GPU available and beneficial
try:
    from tensorflow.keras import mixed_precision
    if gpus:
        mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision enabled (policy: mixed_float16).")
    else:
        print("Mixed precision skipped (no GPU).")
except Exception as e:
    print("Mixed precision not configured:", e)

# Reproducibility seeds (note: fully deterministic training is hard)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


Python, TF versions:
3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
TensorFlow: 2.20.0
No GPU detected.
Mixed precision skipped (no GPU).


In [15]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices('GPU'))


GPUs: []


In [4]:
# Adjust if your data is elsewhere
DATA_ROOT = Path("data_merged")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

# Quick check: class folders and counts
def folder_counts(folder: Path):
    res = {}
    for p in sorted(folder.iterdir()):
        if p.is_dir():
            res[p.name] = sum(1 for _ in p.glob("*") if _.is_file())
    return res

print("Train counts:", folder_counts(TRAIN_DIR))
print("Val counts:  ", folder_counts(VAL_DIR))
print("Test counts: ", folder_counts(TEST_DIR))


Train counts: {'glioma': 2175, 'meningioma': 1407, 'no_tumor': 1362, 'pituitary': 1364}
Val counts:   {'glioma': 62, 'meningioma': 89, 'no_tumor': 29, 'pituitary': 304}
Test counts:  {'glioma': 91, 'meningioma': 139, 'no_tumor': 340, 'pituitary': 108}


In [5]:
IMG_SIZE = 300           # EfficientNetB3 prefers 300
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',          # integer labels -> use SparseCategoricalCrossentropy
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes (order):", class_names)
print("Num classes:", NUM_CLASSES)

# Prefetch & cache for performance
def configure_for_performance(ds, cache=True, training=False):
    if cache:
        ds = ds.cache()
    if training:
        ds = ds.shuffle(2048, seed=SEED)
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = configure_for_performance(train_ds, cache=True, training=True)
val_ds = configure_for_performance(val_ds, cache=True, training=False)
test_ds = configure_for_performance(test_ds, cache=True, training=False)


Found 6308 files belonging to 4 classes.
Found 484 files belonging to 4 classes.
Found 678 files belonging to 4 classes.
Classes (order): ['glioma', 'meningioma', 'no_tumor', 'pituitary']
Num classes: 4


In [6]:
from tensorflow.keras import layers

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.08),
], name="data_augmentation")

# Example preview (optional)
for images, labels in train_ds.take(1):
    aug_images = data_augmentation(images)
    print("Batch shape:", images.shape, "Aug batch shape:", aug_images.shape)
    break


Batch shape: (32, 300, 300, 3) Aug batch shape: (32, 300, 300, 3)


In [7]:
# Compute class counts from the train directory (robust)
train_counts = folder_counts(TRAIN_DIR)
print("Train folder counts:", train_counts)

# Map class_name -> index (same order as train_ds.class_names)
cls_to_index = {cls: idx for idx, cls in enumerate(class_names)}

y_counts = np.array([train_counts.get(cls, 0) for cls in class_names])
total = y_counts.sum()
print("y_counts (in same order):", y_counts)

# Balanced class weights: total / (num_classes * count)
class_weights = {i: (total / (NUM_CLASSES * int(cnt))) if cnt>0 else 1.0 for i,cnt in enumerate(y_counts)}
print("Class weights:", class_weights)


Train folder counts: {'glioma': 2175, 'meningioma': 1407, 'no_tumor': 1362, 'pituitary': 1364}
y_counts (in same order): [2175 1407 1362 1364]
Class weights: {0: np.float64(0.7250574712643678), 1: np.float64(1.1208244491826582), 2: np.float64(1.157856093979442), 3: np.float64(1.156158357771261)}


In [8]:
from tensorflow.keras import regularizers

def build_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout_rate=0.4, l2=1e-4, base_trainable=False):
    # try imagenet weights; fallback to None if offline
    weights = "imagenet"
    try:
        base = keras.applications.EfficientNetB3(
            include_top=False, weights=weights, input_shape=(img_size, img_size, 3), pooling="avg"
        )
    except Exception as e:
        print("Could not load ImageNet weights — falling back to random init:", e)
        base = keras.applications.EfficientNetB3(
            include_top=False, weights=None, input_shape=(img_size, img_size, 3), pooling="avg"
        )

    base.trainable = base_trainable

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)               # augmentation active during training
    x = layers.Conv2D(3, 1, padding="same")(x)  # ensure channels right type if mixed precision
    x = base(x, training=False)
    x = layers.Dropout(dropout_rate)(x)
    # final dense with float32 to avoid mixed precision numerical probs
    outputs = layers.Dense(num_classes, activation="softmax", kernel_regularizer=regularizers.l2(l2), dtype="float32")(x)

    model = keras.Model(inputs, outputs)
    return model, base

model, base = build_model(base_trainable=False)
model.summary()


Could not load ImageNet weights — falling back to random init: Shape mismatch in layer #1 (named stem_conv)for weight stem_conv/kernel. Weight expects shape (3, 3, 1, 40). Received saved weight with shape (3, 3, 3, 40)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_3 (Rescaling)         │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 300, 300, 3)    │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 1536)           │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         6,148 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,789,695 (41.16 MB)

 Trainable params: 6,160 (24.06 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [9]:
# Use sparse categorical (labels are integers)
loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.02)
# If using mixed precision, Adam's default is fine — learning rate scheduling below will help
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("artifacts/best_model_initial.h5", save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.CSVLogger("artifacts/train_log_initial.csv"),
    # keras.callbacks.TensorBoard(log_dir="logs/initial")  # uncomment to use tensorboard
]


In [ ]:
# Use sparse categorical (labels are integers)
loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.02)
# If using mixed precision, Adam's default is fine — learning rate scheduling below will help
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("artifacts/best_model_initial.h5", save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.CSVLogger("artifacts/train_log_initial.csv"),
    # keras.callbacks.TensorBoard(log_dir="logs/initial")  # uncomment to use tensorboard
]


In [10]:
EPOCHS_HEAD = 7

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights,
    callbacks=callbacks
)


Epoch 1/7
  1/198 ━━━━━━━━━━━━━━━━━━━━ 2:50:04 52s/step - accuracy: 0.4062 - loss: 1.2943

AbortedError: Graph execution error:

Detected at node StatefulPartitionedCall/functional_1_1/efficientnetb3_1/block7b_expand_conv_1/convolution defined at (most recent call last):
<stack traces unavailable>
Operation received an exception:Status: 1, message: could not create a memory object, in file tensorflow/core/kernels/mkl/mkl_conv_ops.cc:1112
	 [[{{node StatefulPartitionedCall/functional_1_1/efficientnetb3_1/block7b_expand_conv_1/convolution}}]] [Op:__inference_multi_step_on_iterator_31564]

In [ ]:
# Unfreeze some top layers of the base model for fine-tuning
# Typical: unfreeze last 30-50 layers depending on depth
for layer in base.layers:
    layer.trainable = False
# make last N layers trainable
N = 60
if N > 0:
    for layer in base.layers[-N:]:
        layer.trainable = True

# Recompile with lower LR
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.01),
    metrics=["accuracy"]
)

callbacks_ft = [
    keras.callbacks.ModelCheckpoint("artifacts/best_model_finetuned.h5", save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.CSVLogger("artifacts/train_log_finetune.csv"),
]

EPOCHS_FINETUNE = 20

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weights,
    callbacks=callbacks_ft
)


In [ ]:
def plot_history(h1, h2=None):
    plt.figure(figsize=(14,5))
    # accuracy
    plt.subplot(1,2,1)
    for h,label in [(h1,"head"), (h2,"ft")]:
        if h is None: continue
        acc = h.history.get("accuracy")
        val_acc = h.history.get("val_accuracy")
        epochs = range(1, len(acc)+1)
        plt.plot(epochs, acc, marker='o', label=f"train_{label}")
        plt.plot(epochs, val_acc, marker='x', label=f"val_{label}")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.legend()

    # loss
    plt.subplot(1,2,2)
    for h,label in [(h1,"head"), (h2,"ft")]:
        if h is None: continue
        loss = h.history.get("loss")
        val_loss = h.history.get("val_loss")
        epochs = range(1, len(loss)+1)
        plt.plot(epochs, loss, marker='o', label=f"train_{label}")
        plt.plot(epochs, val_loss, marker='x', label=f"val_{label}")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.legend()
    plt.show()

plot_history(history_head, history_ft)


In [ ]:
# Load best finetuned model if you want:
# model = keras.models.load_model("artifacts/best_model_finetuned.h5")

# Predict
y_true = []
y_pred = []
y_probs = []

for images, labels in tqdm(test_ds, desc="Predicting"):
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(preds.tolist())
    y_probs.extend(probs.tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

# Metrics
from sklearn.metrics import classification_report, confusion_matrix
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

# Plot confusion matrix
import seaborn as sns
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# Save full model (saved_model format recommended)
model.save("artifacts/final_saved_model")
print("Saved to artifacts/final_saved_model")

# To load later:
# loaded = tf.keras.models.load_model("artifacts/final_saved_model")
